# ДЗ 3. Завантаження даних та основи SQL. DQL-команди

**Датасет:** UCI Adult (Census Income), id=2, https://archive.ics.uci.edu/dataset/2/adult

**Задача:** бінарна класифікація `income > 50K`.

## Завдання 1. Setup і завантаження

In [1]:
%pip install -q pgserver psycopg2-binary sqlalchemy pandas pyarrow ucimlrepo

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import pandas as pd
import pgserver
from sqlalchemy import create_engine, text

pg = pgserver.get_server('/tmp/hw3_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri().replace('postgresql://', 'postgresql+psycopg2://', 1), future=True)

with engine.connect() as conn:
    version = conn.execute(text('SELECT version()')).scalar_one()
print(version)

def run(sql):
    with engine.begin() as conn:
        conn.execute(text(sql))

def q(sql):
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn)

PostgreSQL 16.2 on aarch64-apple-darwin23.5.0, compiled by Apple clang version 15.0.0 (clang-1500.0.40.1), 64-bit


In [3]:
from ucimlrepo import fetch_ucirepo

dataset = fetch_ucirepo(id=2)
df = pd.concat([dataset.data.features, dataset.data.targets], axis=1)

data_dir = '/content' if os.path.isdir('/content') else 'data'
os.makedirs(data_dir, exist_ok=True)
csv_path = os.path.join(data_dir, 'hw3_adult.csv')
df.to_csv(csv_path, index=False)

print(df.shape)
df.head()

(48842, 15)


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


### Staging-таблиця (усі колонки TEXT)

In [4]:
run('''
DROP TABLE IF EXISTS hw3_adult CASCADE;
DROP TABLE IF EXISTS hw3_adult_staging CASCADE;

CREATE TABLE hw3_adult_staging (
    age_raw TEXT,
    workclass_raw TEXT,
    fnlwgt_raw TEXT,
    education_raw TEXT,
    education_num_raw TEXT,
    marital_status_raw TEXT,
    occupation_raw TEXT,
    relationship_raw TEXT,
    race_raw TEXT,
    sex_raw TEXT,
    capital_gain_raw TEXT,
    capital_loss_raw TEXT,
    hours_per_week_raw TEXT,
    native_country_raw TEXT,
    income_raw TEXT
);
''')

In [5]:
raw_conn = engine.raw_connection()
try:
    with raw_conn.cursor() as cur:
        with open(csv_path, 'r', encoding='utf-8', newline='') as file:
            cur.copy_expert(
                '''
                COPY hw3_adult_staging
                FROM STDIN
                WITH (FORMAT csv, HEADER true, DELIMITER ',')
                ''',
                file
            )
    raw_conn.commit()
finally:
    raw_conn.close()

q('SELECT COUNT(*) AS loaded_rows FROM hw3_adult_staging')

,loaded_rows
0,48842


## Завдання 2. Staging і DDL

| Колонка | Семантика | Джерело | Тип | Очищення |
|---|---|---|---|---|
| `person_id` | технічний ключ | — | `BIGINT IDENTITY PK` | генерується |
| `age` | вік, років | ціле | `SMALLINT NOT NULL`, CHECK 16–100 | `TRIM`, cast |
| `workclass`, `occupation`, `native_country` | категорії з пропусками | текст, `?` або порожньо | `TEXT` | `?` і `''` → `NULL` |
| `education`, `marital_status`, `relationship`, `race` | категорії без пропусків | текст | `TEXT NOT NULL` | `TRIM` |
| `education_num` | порядковий рівень освіти | ціле | `SMALLINT`, CHECK 1–16 | cast |
| `sex` | стать | `Male`/`Female` | `TEXT`, CHECK IN | `TRIM` |
| `fnlwgt` | вага вибірки перепису | ціле | `INTEGER`, CHECK > 0 | cast |
| `capital_gain`, `capital_loss` | грошові суми, USD | ціле | `NUMERIC(12,2)`, CHECK ≥ 0 | cast |
| `hours_per_week` | робочих годин на тиждень | ціле | `SMALLINT`, CHECK 1–99 | cast |
| `income_gt_50k` | target | `<=50K`, `>50K`, з крапкою або без | `BOOLEAN NOT NULL` | прибрати `.`, порівняти з `>50K` |

У датасеті немає дат і моментів часу, тому `DATE`/`TIMESTAMPTZ` не застосовуються. Довільні тексти зберігаються як `TEXT`, бо бізнес-ліміту довжини немає.

In [6]:
run('''
CREATE TABLE hw3_adult (
    person_id BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    age SMALLINT NOT NULL CHECK (age BETWEEN 16 AND 100),
    workclass TEXT,
    fnlwgt INTEGER NOT NULL CHECK (fnlwgt > 0),
    education TEXT NOT NULL,
    education_num SMALLINT NOT NULL CHECK (education_num BETWEEN 1 AND 16),
    marital_status TEXT NOT NULL,
    occupation TEXT,
    relationship TEXT NOT NULL,
    race TEXT NOT NULL,
    sex TEXT NOT NULL CHECK (sex IN ('Male', 'Female')),
    capital_gain NUMERIC(12, 2) NOT NULL CHECK (capital_gain >= 0),
    capital_loss NUMERIC(12, 2) NOT NULL CHECK (capital_loss >= 0),
    hours_per_week SMALLINT NOT NULL CHECK (hours_per_week BETWEEN 1 AND 99),
    native_country TEXT,
    income_gt_50k BOOLEAN NOT NULL
);
''')

In [7]:
run('''
WITH parsed AS (
    SELECT
        NULLIF(TRIM(age_raw), '')::SMALLINT AS age,
        NULLIF(NULLIF(TRIM(workclass_raw), ''), '?') AS workclass,
        NULLIF(TRIM(fnlwgt_raw), '')::INTEGER AS fnlwgt,
        TRIM(education_raw) AS education,
        NULLIF(TRIM(education_num_raw), '')::SMALLINT AS education_num,
        TRIM(marital_status_raw) AS marital_status,
        NULLIF(NULLIF(TRIM(occupation_raw), ''), '?') AS occupation,
        TRIM(relationship_raw) AS relationship,
        TRIM(race_raw) AS race,
        TRIM(sex_raw) AS sex,
        NULLIF(TRIM(capital_gain_raw), '')::NUMERIC(12, 2) AS capital_gain,
        NULLIF(TRIM(capital_loss_raw), '')::NUMERIC(12, 2) AS capital_loss,
        NULLIF(TRIM(hours_per_week_raw), '')::SMALLINT AS hours_per_week,
        NULLIF(NULLIF(TRIM(native_country_raw), ''), '?') AS native_country,
        RTRIM(TRIM(income_raw), '.') AS income
    FROM hw3_adult_staging
)
INSERT INTO hw3_adult (
    age, workclass, fnlwgt, education, education_num, marital_status,
    occupation, relationship, race, sex, capital_gain, capital_loss,
    hours_per_week, native_country, income_gt_50k
)
SELECT
    age, workclass, fnlwgt, education, education_num, marital_status,
    occupation, relationship, race, sex, capital_gain, capital_loss,
    hours_per_week, native_country, income = '>50K'
FROM parsed
WHERE age IS NOT NULL
  AND income IN ('>50K', '<=50K');
''')

## Завдання 3. Data-quality audit

### 3.1. Контроль кількості рядків

In [8]:
q('''
SELECT 'staging' AS layer, COUNT(*) AS n_rows FROM hw3_adult_staging
UNION ALL
SELECT 'clean' AS layer, COUNT(*) AS n_rows FROM hw3_adult;
''')

,layer,n_rows
0,staging,48842
1,clean,48842


**Висновок:** staging і clean містять по 48 842 рядки. Фільтр `age IS NOT NULL AND income IN ('>50K', '<=50K')` не відкинув жодного рядка: після прибирання крапки (`>50K.`) усі мітки валідні, а пропуски в категоріях переведено в `NULL`, а не видалено.

### 3.2. Перевірка дублікатів

In [9]:
q('''
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (age, workclass, fnlwgt, education, marital_status, occupation,
                    relationship, race, sex, capital_gain, capital_loss,
                    hours_per_week, native_country, income_gt_50k)) AS distinct_business_keys,
    COUNT(*) - COUNT(DISTINCT (age, workclass, fnlwgt, education, marital_status, occupation,
                               relationship, race, sex, capital_gain, capital_loss,
                               hours_per_week, native_country, income_gt_50k)) AS duplicate_rows
FROM hw3_adult;
''')

,total_rows,distinct_business_keys,duplicate_rows
0,48842,48790,52


**Висновок:** природного ключа в Adult немає, тому business key — комбінація всіх 14 ознак. Знайдено 52 потенційні дублікати (0,1%). Це можуть бути різні люди з однаковим профілем, тому рядки не видаляються, але перед навчанням варто перевірити, чи не потрапляють копії одночасно в train і test.

### 3.3. Null-rate і спеціальні маркери

In [10]:
q('''
SELECT
    ROUND(COUNT(*) FILTER (WHERE workclass IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_workclass,
    ROUND(COUNT(*) FILTER (WHERE occupation IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_occupation,
    ROUND(COUNT(*) FILTER (WHERE native_country IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_native_country,
    ROUND(COUNT(*) FILTER (WHERE age IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_age,
    ROUND(COUNT(*) FILTER (WHERE education IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_education,
    ROUND(COUNT(*) FILTER (WHERE hours_per_week IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_hours,
    COUNT(*) AS total_rows
FROM hw3_adult;
''')

,pct_null_workclass,pct_null_occupation,pct_null_native_country,pct_null_age,pct_null_education,pct_null_hours,total_rows
0,5.73,5.75,1.75,0.0,0.0,0.0,48842


In [11]:
q('''
SELECT
    COUNT(*) FILTER (WHERE TRIM(workclass_raw) = '?') AS workclass_question,
    COUNT(*) FILTER (WHERE workclass_raw IS NULL OR TRIM(workclass_raw) = '') AS workclass_empty,
    COUNT(*) FILTER (WHERE TRIM(occupation_raw) = '?') AS occupation_question,
    COUNT(*) FILTER (WHERE occupation_raw IS NULL OR TRIM(occupation_raw) = '') AS occupation_empty,
    COUNT(*) FILTER (WHERE TRIM(native_country_raw) = '?') AS country_question,
    COUNT(*) FILTER (WHERE native_country_raw IS NULL OR TRIM(native_country_raw) = '') AS country_empty,
    COUNT(*) FILTER (WHERE income_raw LIKE '%.') AS income_with_dot,
    COUNT(*) FILTER (WHERE LOWER(TRIM(workclass_raw)) = 'unknown') AS workclass_unknown
FROM hw3_adult_staging;
''')

,workclass_question,workclass_empty,occupation_question,occupation_empty,country_question,country_empty,income_with_dot,workclass_unknown
0,1836,963,1843,966,583,274,16281,0


**Висновок:** пропуски є лише в `workclass` (5,73%), `occupation` (5,75%) і `native_country` (1,75%); `age`, `education`, `hours_per_week` повні. У сирих даних пропуск закодовано двома способами: `?` (1836 / 1843 / 583) і порожнім значенням (963 / 966 / 274), що варто уніфікувати в джерелі. Також 16 281 мітка `income` має зайву крапку (частина test-файлу UCI). Маркер `unknown` відсутній. Нульовий null-rate інших колонок не гарантує якості: наприклад, `capital_gain = 99999` — прихований sentinel.

## Завдання 4. Розподіли та потенційні аномалії

### 4.1. Розподіл категорії `occupation`

In [12]:
q('''
SELECT
    occupation,
    COUNT(*) AS n_rows,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_rows
FROM hw3_adult
GROUP BY occupation
ORDER BY n_rows DESC, occupation
LIMIT 10;
''')

,occupation,n_rows,pct_rows
0,Prof-specialty,6172,12.64
1,Craft-repair,6112,12.51
2,Exec-managerial,6086,12.46
3,Adm-clerical,5611,11.49
4,Sales,5504,11.27
5,Other-service,4923,10.08
6,Machine-op-inspct,3022,6.19
7,NaN,2809,5.75
8,Transport-moving,2355,4.82
9,Handlers-cleaners,2072,4.24


**Висновок:** розподіл `occupation` досить рівномірний: шість найбільших категорій мають по 10–13%. Пропуски (`NULL`, 5,75%) — восьма за розміром група, тому для моделі їх варто кодувати окремою категорією, а не відкидати.

### 4.2. Статистика `hours_per_week`

In [13]:
q('''
SELECT
    COUNT(hours_per_week) AS n_non_null,
    MIN(hours_per_week) AS min_value,
    MAX(hours_per_week) AS max_value,
    ROUND(AVG(hours_per_week)::NUMERIC, 2) AS avg_value,
    ROUND(STDDEV_SAMP(hours_per_week)::NUMERIC, 2) AS sd_value
FROM hw3_adult;
''')

,n_non_null,min_value,max_value,avg_value,sd_value
0,48842,1,99,40.42,12.39


**Висновок:** середнє 40,42 год при SD 12,39; мінімум 1 і максимум 99. Значення 99 схоже на верхню межу шкали анкети, а не на реальні 99 годин.

### 4.3. Domain outlier-check: sentinel-значення

In [14]:
q('''
SELECT
    COUNT(*) FILTER (WHERE capital_gain = 99999) AS capital_gain_99999,
    COUNT(*) FILTER (WHERE hours_per_week = 99) AS hours_99,
    COUNT(*) FILTER (WHERE age = 90) AS age_90,
    COUNT(*) FILTER (WHERE capital_gain > 0 AND capital_loss > 0) AS gain_and_loss
FROM hw3_adult;
''')

,capital_gain_99999,hours_99,age_90,gain_and_loss
0,244,137,55,0


**Висновок:** 244 рядки мають `capital_gain = 99999`, що є технічною межею кодування (top-coding). 137 рядків з `hours_per_week = 99` і 55 з `age = 90` теж виглядають як верхні межі шкали. Рядків, де одночасно є gain і loss, немає. **Дія:** не видаляти, а додати прапорець `is_top_coded` і за потреби обрізати значення для лінійних моделей.

## Завдання 5. Десять DQL/EDA-запитів

**Business-question:** Яка частка доходу >50K для кожного рівня освіти?

In [15]:
q("""
SELECT
    education,
    education_num,
    COUNT(*) AS n_rows,
    ROUND(AVG(income_gt_50k::INT) * 100, 2) AS pct_gt_50k
FROM hw3_adult
GROUP BY education, education_num
ORDER BY education_num DESC
LIMIT 16;
""")

,education,education_num,n_rows,pct_gt_50k
0,Doctorate,16,594,72.56
1,Prof-school,15,834,73.98
2,Masters,14,2657,54.91
3,Bachelors,13,8025,41.28
4,Assoc-acdm,12,1601,25.80
5,Assoc-voc,11,2061,25.33
6,Some-college,10,10878,18.96
7,HS-grad,9,15784,15.86
8,12th,8,657,7.31
9,11th,7,1812,5.08


**Interpretation:** Частка доходу >50K монотонно зростає з рівнем освіти: від 1,2% (Preschool) і 15,9% (HS-grad) до 41,3% (Bachelors) і понад 72% (Prof-school, Doctorate). `education` і `education_num` дублюють одну інформацію, тому в модель варто брати лише одну з них.

**Business-question:** Які workclass найбільше пов'язані з пропусками occupation?

In [16]:
q("""
SELECT
    COALESCE(workclass, '(missing)') AS workclass,
    COUNT(*) AS n_rows,
    COUNT(*) FILTER (WHERE occupation IS NULL) AS missing_occupation
FROM hw3_adult
GROUP BY COALESCE(workclass, '(missing)')
ORDER BY missing_occupation DESC, n_rows DESC;
""")

,workclass,n_rows,missing_occupation
0,(missing),2799,2799
1,Never-worked,10,10
2,Private,33906,0
3,Self-emp-not-inc,3862,0
4,Local-gov,3136,0
5,State-gov,1981,0
6,Self-emp-inc,1695,0
7,Federal-gov,1432,0
8,Without-pay,21,0


**Interpretation:** Усі 2799 рядків без `workclass` також не мають `occupation`, а ще 10 пропусків `occupation` припадають на `Never-worked`. Отже, пропуски не випадкові: вони відповідають людям без роботи або з незаповненим блоком про зайнятість, і це варто передати моделі як окрему ознаку.

**Business-question:** Скільки жінок у самозайнятості або на федеральній службі мають дохід >50K, не будучи заміжніми?

In [17]:
q("""
SELECT
    workclass,
    COUNT(*) AS n_rows
FROM hw3_adult
WHERE sex = 'Female'
  AND (workclass = 'Federal-gov' OR workclass LIKE 'Self-emp%')
  AND NOT (marital_status LIKE 'Married%')
  AND income_gt_50k
GROUP BY workclass
ORDER BY n_rows DESC;
""")

,workclass,n_rows
0,Federal-gov,38
1,Self-emp-not-inc,38
2,Self-emp-inc,21


**Interpretation:** Серед незаміжніх жінок з доходом >50K у цих секторах найбільше `Federal-gov` і `Self-emp-not-inc` (по 38), `Self-emp-inc` — 21. Групи малі, тому висновки про них статистично нестабільні.

**Business-question:** Як відрізняється частка >50K у людей зі стандартним тижнем 35–45 годин і поза ним?

In [18]:
q("""
SELECT
    hours_per_week BETWEEN 35 AND 45 AS standard_week,
    COUNT(*) AS n_rows,
    ROUND(AVG(income_gt_50k::INT) * 100, 2) AS pct_gt_50k
FROM hw3_adult
GROUP BY hours_per_week BETWEEN 35 AND 45
ORDER BY standard_week DESC;
""")

,standard_week,n_rows,pct_gt_50k
0,True,29746,22.37
1,False,19096,26.35


**Interpretation:** Люди зі стандартним тижнем 35–45 год мають частку >50K 22,4%, а поза ним — 26,4%. Різниця невелика, бо до «поза» потрапляють і ті, хто працює неповний день (низький дохід), і ті, хто працює понад 45 год (високий дохід). Тому годин краще кодувати кількома інтервалами.

**Business-question:** Яка частка >50K у різних групах сімейного стану з переліку?

In [19]:
q("""
SELECT
    marital_status,
    COUNT(*) AS n_rows,
    ROUND(AVG(income_gt_50k::INT) * 100, 2) AS pct_gt_50k
FROM hw3_adult
WHERE marital_status IN ('Married-civ-spouse', 'Never-married', 'Divorced', 'Widowed')
GROUP BY marital_status
ORDER BY pct_gt_50k DESC;
""")

,marital_status,n_rows,pct_gt_50k
0,Married-civ-spouse,22379,44.61
1,Divorced,6633,10.12
2,Widowed,1518,8.43
3,Never-married,16117,4.55


**Interpretation:** Найвища частка >50K у `Married-civ-spouse` (44,6%), найнижча у `Never-married` (4,6%). Ймовірно, це пов'язано з віком і сукупним доходом домогосподарства, тому ознака сильна, але потребує обережної інтерпретації.

**Business-question:** Скільки різних країн походження представлено і які з них рідкісні (менше 10 записів)?

In [20]:
q("""
SELECT
    native_country,
    COUNT(*) AS n_rows
FROM hw3_adult
WHERE native_country IS NOT NULL
GROUP BY native_country
HAVING COUNT(*) < 10
ORDER BY n_rows, native_country;
""")

,native_country,n_rows
0,Holand-Netherlands,1


**Interpretation:** Лише `Holand-Netherlands` має менше 10 записів (1 рядок). Таку категорію варто об'єднати в `Other`, щоб уникнути проблем із кодуванням та розбиттям train/test.

**Business-question:** Скільки унікальних країн у датасеті?

In [21]:
q("""
SELECT COUNT(DISTINCT native_country) AS n_countries
FROM hw3_adult;
""")

,n_countries
0,41


**Interpretation:** Датасет містить 41 відому країну. Більшість записів — США, тому `native_country` доцільно спростити до `US` / `non-US` або до регіонів.

**Business-question:** Скільки людей не з США: порівняння `<>` та `IS DISTINCT FROM` за наявності NULL.

In [22]:
q("""
SELECT
    COUNT(*) FILTER (WHERE native_country <> 'United-States') AS not_us_by_neq,
    COUNT(*) FILTER (WHERE native_country IS DISTINCT FROM 'United-States') AS not_us_by_distinct,
    COUNT(*) FILTER (WHERE native_country IS NULL) AS country_null
FROM hw3_adult;
""")

,not_us_by_neq,not_us_by_distinct,country_null
0,4153,5010,857


**Interpretation:** `<>` повертає 4153 рядки, а `IS DISTINCT FROM` — 5010. Різниця, 857, точно дорівнює кількості `NULL`: звичайне порівняння з `NULL` дає UNKNOWN і рядок випадає. Для фільтрів з пропусками потрібно явно обирати потрібну семантику.

**Business-question:** Який середній ненульовий capital_gain у групах доходу?

In [23]:
q("""
SELECT
    income_gt_50k,
    COUNT(NULLIF(capital_gain, 0)) AS n_with_gain,
    ROUND(AVG(NULLIF(capital_gain, 0)), 2) AS avg_nonzero_gain
FROM hw3_adult
GROUP BY income_gt_50k
ORDER BY income_gt_50k;
""")

,income_gt_50k,n_with_gain,avg_nonzero_gain
0,False,1544,3537.67
1,True,2491,18964.93


**Interpretation:** Серед людей із ненульовим `capital_gain` середня сума в групі >50K становить 18 965, а в групі <=50K — 3 538. Ознака сильна, але більшість значень нульові, тому краще використовувати її разом із бінарним прапорцем `has_capital_gain`.

**Business-question:** Хто 10 найстарших людей з доходом >50K, що працюють понад 60 годин?

In [24]:
q("""
SELECT DISTINCT
    age,
    occupation,
    hours_per_week,
    education
FROM hw3_adult
WHERE income_gt_50k
  AND hours_per_week > 60
  AND occupation IS NOT NULL
ORDER BY age DESC, hours_per_week DESC
LIMIT 10;
""")

,age,occupation,hours_per_week,education
0,90,Prof-specialty,72,Prof-school
1,73,Farming-fishing,99,7th-8th
2,71,Exec-managerial,75,11th
3,70,Sales,70,Bachelors
4,70,Sales,65,Some-college
5,69,Craft-repair,72,Some-college
6,68,Sales,70,Assoc-acdm
7,67,Exec-managerial,75,Bachelors
8,67,Exec-managerial,72,Masters
9,66,Exec-managerial,86,HS-grad


**Interpretation:** Найстарша людина з високим доходом і понад 60 год роботи має 90 років (верхня межа шкали віку), а 73-річний фермер має 99 год — теж межа. Серед решти переважають `Exec-managerial` і `Sales`, що відповідає професіям із високим навантаженням.

## Завдання 6. Reflection

Основні проблеми якості в Adult: пропуски в `workclass`, `occupation` і `native_country`, закодовані двома різними способами (`?` і порожнє значення); непослідовні мітки target (`>50K` і `>50K.`); top-coded значення (`capital_gain = 99999`, `hours_per_week = 99`, `age = 90`); 52 потенційні дублікати; рідкісні категорії на кшталт `Holand-Netherlands`. Пропуски в `workclass` і `occupation` збігаються, тобто вони не випадкові.

Як features я використав би `age`, `education_num`, `marital_status`, `occupation`, `workclass`, `hours_per_week`, `capital_gain`, `capital_loss` і спрощену `native_country`. `education` не потрібна, бо дублює `education_num`. `fnlwgt` — вага вибірки перепису, а не властивість людини, тому вона доречна як sample weight, але не як feature. `sex` і `race` потребують аудиту fairness, а не автоматичного включення.

Прямого leakage немає, бо всі ознаки описують людину на момент анкети. Проте `capital_gain` і `capital_loss` фіксуються за той самий рік, що й дохід, тож якщо прогноз робиться до кінця року, вони можуть бути ще невідомими.

Data engineer я запропонував би єдиний маркер пропуску (`NULL`), однакову нотацію target у train і test, явні прапорці top-coding, словник категорій із контролем рідкісних значень і стабільний ідентифікатор запису для перевірки дублікатів.

Датасет природно відповідає задачі бінарної класифікації `income > 50K` з дисбалансом класів приблизно 76/24. Через цей дисбаланс для оцінки моделі доречніші ROC-AUC, PR-AUC і F1, ніж accuracy, а розбиття train/test варто робити стратифіковано за target.